In [5]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv('Day16_Student_Wellbeing_Survey.csv')

print("Columns:", df.columns.tolist())
print("Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())

Columns: ['Student_ID', 'Age', 'Faculty', 'Year_of_Study', 'City', 'Accommodation', 'Scholarship', 'Part_Time_Job', 'Internet_Quality', 'Preferred_Study_Space', 'Weekly_Study_Hours', 'Average_Sleep_Hours', 'Daily_Screen_Time_Hours', 'Exercise_Days_Per_Week', 'Commute_Time_Minutes', 'Stress_Score', 'Academic_Readiness_Score', 'Overall_Satisfaction', 'Monthly_Discretionary_Spending', 'Social_Activity_Hours_Per_Week']
Shape: (600, 20)

First 5 rows:
  Student_ID  Age        Faculty  Year_of_Study        City     Accommodation  \
0    STU0001   21   Data Science              3   Bengaluru            Hostel   
1    STU0002   21   Data Science              4        Pune            Hostel   
2    STU0003   22  Life Sciences              2  Chandigarh              Home   
3    STU0004   18       Business              4   Bengaluru            Hostel   
4    STU0005   20       Business              3       Jammu  Shared Apartment   

  Scholarship Part_Time_Job Internet_Quality Preferred_Study_S

In [6]:
from scipy import stats

vars_5 = ['Weekly_Study_Hours', 'Average_Sleep_Hours', 'Daily_Screen_Time_Hours', 'Stress_Score', 'Academic_Readiness_Score']

print("--- CENTRAL TENDENCY & DISPERSION ---")
summary_dict = []
for v in vars_5:
    s = df[v]
    mean_val = s.mean()
    median_val = s.median()
    mode_val = s.mode()[0]
    range_val = s.max() - s.min()
    var_sample = s.var(ddof=1)
    std_sample = s.std(ddof=1)
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    cv = (std_sample / mean_val) * 100

    summary_dict.append({
        'Variable': v,
        'Mean': round(mean_val, 4),
        'Median': round(median_val, 4),
        'Mode': round(mode_val, 4),
        'Range': round(range_val, 4),
        'Variance (Sample)': round(var_sample, 4),
        'Std Dev (Sample)': round(std_sample, 4),
        'Q1': round(q1, 4),
        'Q3': round(q3, 4),
        'IQR': round(iqr, 4),
        'CV (%)': round(cv, 2)
    })

df_summary = pd.DataFrame(summary_dict)
print(df_summary.to_string())

--- CENTRAL TENDENCY & DISPERSION ---
                   Variable     Mean  Median  Mode  Range  Variance (Sample)  Std Dev (Sample)      Q1      Q3     IQR  CV (%)
0        Weekly_Study_Hours  15.6910   15.40  13.1   30.0            19.2170            4.3837  13.000  18.425   5.425   27.94
1       Average_Sleep_Hours   6.9975    7.00   7.0    5.0             0.7034            0.8387   6.475   7.600   1.125   11.99
2   Daily_Screen_Time_Hours   4.5030    4.20   3.5   11.2             3.4690            1.8625   3.200   5.400   2.200   41.36
3              Stress_Score   4.4645    4.50   4.7    8.4             2.7825            1.6681   3.300   5.700   2.400   37.36
4  Academic_Readiness_Score  71.7732   71.65  71.1   56.1            93.9820            9.6944  65.200  78.025  12.825   13.51


In [7]:
outlier_vars = ['Weekly_Study_Hours', 'Daily_Screen_Time_Hours', 'Commute_Time_Minutes', 'Monthly_Discretionary_Spending']

print("--- OUTLIER DETECTION VIA IQR METHOD ---")
for v in outlier_vars:
    s = df[v]
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outliers = s[(s < lower_bound) | (s > upper_bound)]
    print(f"\nVariable: {v}")
    print(f"  Q1: {q1:.2f}, Q3: {q3:.2f}, IQR: {iqr:.2f}")
    print(f"  Lower Bound: {lower_bound:.2f}, Upper Bound: {upper_bound:.2f}")
    print(f"  Count of Outliers: {len(outliers)}")
    if len(outliers) > 0:
        print(f"  Outlier Values: {outliers.values}")

    s_clean = s[(s >= lower_bound) & (s <= upper_bound)]
    print(f"  Before Removal -> Mean: {s.mean():.4f}, Median: {s.median():.4f}")
    print(f"  After Removal  -> Mean: {s_clean.mean():.4f}, Median: {s_clean.median():.4f}")

--- OUTLIER DETECTION VIA IQR METHOD ---

Variable: Weekly_Study_Hours
  Q1: 13.00, Q3: 18.42, IQR: 5.42
  Lower Bound: 4.86, Upper Bound: 26.56
  Count of Outliers: 8
  Outlier Values: [27.8 26.6 26.6  4.  27.2 34.   4.3 27.2]
  Before Removal -> Mean: 15.6910, Median: 15.4000
  After Removal  -> Mean: 15.6029, Median: 15.3500

Variable: Daily_Screen_Time_Hours
  Q1: 3.20, Q3: 5.40, IQR: 2.20
  Lower Bound: -0.10, Upper Bound: 8.70
  Count of Outliers: 18
  Outlier Values: [ 9.8 10.1 11.8 10.4 12.4 12.  10.6 11.3  9.1 12.  10.4  9.2 11.  11.1
  9.3  9.  12.   9.9]
  Before Removal -> Mean: 4.5030, Median: 4.2000
  After Removal  -> Mean: 4.3134, Median: 4.1000

Variable: Commute_Time_Minutes
  Q1: 13.38, Q3: 31.52, IQR: 18.15
  Lower Bound: -13.85, Upper Bound: 58.75
  Count of Outliers: 4
  Outlier Values: [ 65.2  62.3  72.5 118. ]
  Before Removal -> Mean: 22.8938, Median: 20.4500
  After Removal  -> Mean: 22.5139, Median: 20.3500

Variable: Monthly_Discretionary_Spending
  Q1: 4113

In [8]:
N = len(df)

# Defined events
mask_A = df['Part_Time_Job'] == 'Yes'
mask_B = df['Stress_Score'] >= 7
mask_C = df['Scholarship'] == 'Yes'
mask_D = df['Exercise_Days_Per_Week'] >= 3

# Individual counts
nA = mask_A.sum()
nB = mask_B.sum()
nC = mask_C.sum()
nD = mask_D.sum()

# Probabilities
pA = nA / N
pB = nB / N
pC = nC / N
pD = nD / N

# Union and Intersection
mask_A_or_B = mask_A | mask_B
mask_A_and_B = mask_A & mask_B

nA_or_B = mask_A_or_B.sum()
nA_and_B = mask_A_and_B.sum()

pA_or_B = nA_or_B / N
pA_and_B = nA_and_B / N

# Conditionals
pA_given_B = nA_and_B / nB
pB_given_A = nA_and_B / nA

print(f"Total N = {N}")
print(f"n(A) = {nA}, P(A) = {pA:.4f}")
print(f"n(B) = {nB}, P(B) = {pB:.4f}")
print(f"n(C) = {nC}, P(C) = {pC:.4f}")
print(f"n(D) = {nD}, P(D) = {pD:.4f}")
print(f"n(A or B) = {nA_or_B}, P(A or B) = {pA_or_B:.4f}")
print(f"n(A and B) = {nA_and_B}, P(A and B) = {pA_and_B:.4f}")
print(f"P(A|B) = {pA_given_B:.4f}")
print(f"P(B|A) = {pB_given_A:.4f}")

# Mutually Exclusive Check for Year_of_Study == 1 and Year_of_Study == 4
y1 = df['Year_of_Study'] == 1
y4 = df['Year_of_Study'] == 4
intersection_y1_y4 = (y1 & y4).sum()
print(f"\nYear 1 AND Year 4 count = {intersection_y1_y4}")
print(f"Are Year 1 and Year 4 mutually exclusive? {intersection_y1_y4 == 0}")

# Independence check: P(A and B) vs P(A) * P(B)
pA_times_pB = pA * pB
print(f"\nP(A and B) = {pA_and_B:.6f}")
print(f"P(A) * P(B) = {pA_times_pB:.6f}")
print(f"Difference = {pA_and_B - pA_times_pB:.6f}")

# Bayes' theorem check
# P(A|B) = P(B|A) * P(A) / P(B)
# P(B) can also be expanded as P(B|A)*P(A) + P(B|not A)*P(not A)
mask_not_A = ~mask_A
n_not_A = mask_not_A.sum()
p_not_A = n_not_A / N
nB_and_notA = (mask_B & mask_not_A).sum()
pB_given_notA = nB_and_notA / n_not_A

bayes_pB = (pB_given_A * pA) + (pB_given_notA * p_not_A)
bayes_pA_given_B = (pB_given_A * pA) / bayes_pB

print(f"\n--- BAYES' THEOREM ---")
print(f"P(not A) = {p_not_A:.4f}")
print(f"P(B|not A) = {pB_given_notA:.4f}")
print(f"Expanded P(B) via law of total probability = {bayes_pB:.4f}")
print(f"Bayes calculated P(A|B) = {bayes_pA_given_B:.6f}")
print(f"Direct P(A|B) = {pA_given_B:.6f}")

Total N = 600
n(A) = 152, P(A) = 0.2533
n(B) = 45, P(B) = 0.0750
n(C) = 184, P(C) = 0.3067
n(D) = 356, P(D) = 0.5933
n(A or B) = 166, P(A or B) = 0.2767
n(A and B) = 31, P(A and B) = 0.0517
P(A|B) = 0.6889
P(B|A) = 0.2039

Year 1 AND Year 4 count = 0
Are Year 1 and Year 4 mutually exclusive? True

P(A and B) = 0.051667
P(A) * P(B) = 0.019000
Difference = 0.032667

--- BAYES' THEOREM ---
P(not A) = 0.7467
P(B|not A) = 0.0312
Expanded P(B) via law of total probability = 0.0750
Bayes calculated P(A|B) = 0.688889
Direct P(A|B) = 0.688889


In [9]:
s_ars = df['Academic_Readiness_Score']

mean_ars = s_ars.mean()
std_ars = s_ars.std(ddof=1) # sample std dev
min_val = s_ars.min()
max_val = s_ars.max()

z_min = (min_val - mean_ars) / std_ars
z_max = (max_val - mean_ars) / std_ars

print(f"Academic_Readiness_Score Mean: {mean_ars:.4f}")
print(f"Academic_Readiness_Score Std Dev: {std_ars:.4f}")
print(f"Min Value: {min_val:.2f}, Z-score Min: {z_min:.4f}")
print(f"Max Value: {max_val:.2f}, Z-score Max: {z_max:.4f}")

# Empirical Rule check
within_1sd = df[(s_ars >= mean_ars - std_ars) & (s_ars <= mean_ars + std_ars)]
within_2sd = df[(s_ars >= mean_ars - 2*std_ars) & (s_ars <= mean_ars + 2*std_ars)]
within_3sd = df[(s_ars >= mean_ars - 3*std_ars) & (s_ars <= mean_ars + 3*std_ars)]

pct_1sd = (len(within_1sd) / N) * 100
pct_2sd = (len(within_2sd) / N) * 100
pct_3sd = (len(within_3sd) / N) * 100

print(f"\nEmpirical Rule vs Actual Dataset:")
print(f"Within 1 SD: Expected ~68%, Actual = {pct_1sd:.2f}% ({len(within_1sd)} students)")
print(f"Within 2 SD: Expected ~95%, Actual = {pct_2sd:.2f}% ({len(within_2sd)} students)")
print(f"Within 3 SD: Expected ~99.7%, Actual = {pct_3sd:.2f}% ({len(within_3sd)} students)")

Academic_Readiness_Score Mean: 71.7732
Academic_Readiness_Score Std Dev: 9.6944
Min Value: 41.90, Z-score Min: -3.0815
Max Value: 98.00, Z-score Max: 2.7054

Empirical Rule vs Actual Dataset:
Within 1 SD: Expected ~68%, Actual = 69.50% (417 students)
Within 2 SD: Expected ~95%, Actual = 95.17% (571 students)
Within 3 SD: Expected ~99.7%, Actual = 99.83% (599 students)


In [10]:
min_stu = df[df['Academic_Readiness_Score'] == min_val]
max_stu = df[df['Academic_Readiness_Score'] == max_val]

print("Min student:")
print(min_stu[['Student_ID', 'Academic_Readiness_Score']])

print("Max student:")
print(max_stu[['Student_ID', 'Academic_Readiness_Score']])

Min student:
    Student_ID  Academic_Readiness_Score
542    STU0543                      41.9
Max student:
    Student_ID  Academic_Readiness_Score
130    STU0131                      98.0
284    STU0285                      98.0
432    STU0433                      98.0
